# Dimensionality Reduction

### Key Takeaways:
By the end of the class, you should be able to:

- Understand why dimensionality reduction methods are used
  - PCA
  - tSNE/UMAP
- Derive, analyse and plot PCA-transformed gene expression data
- Apply UMAP and tSNE to a single-cell RNAseq dataset and plot the results using various dimensionality reduction parameters

Website Resources:
  - https://sebastianraschka.com/Articles/2015_pca_in_3_steps.html
  - https://stats.stackexchange.com/questions/2691/making-sense-of-principal-component-analysis-eigenvectors-eigenvalues
  - https://scanpy-tutorials.readthedocs.io/en/latest/index.html
  

In [ ]:
# Start with importing the packages
%matplotlib inline
!pip install umap-learn[plot]

import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from   sklearn.decomposition import PCA
from   sklearn.manifold import TSNE
from   sklearn.preprocessing import StandardScaler
import h5py
from umap import UMAP
import umap.plot
import scanpy as sc
import anndata as ad

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm
import scipy
from scipy import stats
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix


np.set_printoptions(suppress=True)
np.set_printoptions(precision=4)

plt_style = 'seaborn-notebook'
pd.set_option('styler.format.precision', 2)
pd.set_option('styler.render.max_columns',10)

## Data loading

We will start by loading in an RNAseq dataset of normal melanocytes and a melanoma cell line.


In [ ]:
# Read the melanoma dataset
melanoma_rnaseq = pd.read_excel('~/LECTURE_MATERIALS/DataFiles/melanoma_zerosRemoved_log2transformed_2026.xlsx',index_col = 0)
melanoma_rnaseq.head()


In [ ]:
#Save the sample metadata to its own dataframe
melanoma_metadata = melanoma_rnaseq.loc[:,:'cell_line']
melanoma_metadata.head()

# Part 1: PCA


We are going to calculate a PCA transformation of this data and plot it out. Note: our data set consists of 12 samples. We can only calculate a maximum of 12 PCs from this data, reducing our dimensionality from over 15,000 genes.

## Step 1: Standardization (Scale the Data)

The data should be scaled such that the variance is standardized. A common normalization applied prior to PCA is to set the mean of each dimension to 0 and the standard deviation to 1. This can be applied simply in python using the `StandardScaler().fit_transform()` function.



In [ ]:
#For PCA, we need to scale and center the data
#The mean of each gene expression is set to 0 and the standard deviation to 1
melanoma_scaled = StandardScaler().fit_transform(melanoma_rnaseq.loc[:, 'A1BG':])
display(melanoma_scaled)

In [ ]:
# To give a sense for what StandardScaler() is doing, let's summarize the parameters of the scaled data
pd.DataFrame(data=melanoma_scaled).iloc[:,:5].describe()

## Step 2: Calculate the PCA model

The resulting PCA model enables 3 main outputs:
 - The principal components (the relative contribution of each dimension (gene) in each principal component)
 - The prinicipal component-transformed data
 - The explained variance (the **eigenvalues** of the covariance matrix)

In [ ]:
#Now, we calculate the PCA transform and apply it to the data

# Setting up the PCA model
melanoma_pca = PCA().fit(melanoma_scaled)

# PCA transform data
melanoma_pca_data = melanoma_pca.transform(melanoma_scaled)



The PCA transform has been calculated, so we can plot it.



In [ ]:
# Plotting the PCA-transformed data
fig = plt.figure()
ax = fig.add_subplot(111)


ax.scatter(melanoma_pca_data[melanoma_metadata.cell_line == 'FM', 0],
          melanoma_pca_data[melanoma_metadata.cell_line == 'FM', 1], alpha=0.4,c='b', label="FM (normal)")
ax.scatter(melanoma_pca_data[melanoma_metadata.cell_line == 'SK_MEL_28', 0],
          melanoma_pca_data[melanoma_metadata.cell_line == 'SK_MEL_28', 1], alpha=0.4,c='r', label='SK_MEL_28')
ax.scatter(melanoma_pca_data[melanoma_metadata.cell_line == 'SK_MEL_147', 0],
          melanoma_pca_data[melanoma_metadata.cell_line == 'SK_MEL_147', 1], alpha=0.4,c='m', label='SK_MEL_147')
ax.scatter(melanoma_pca_data[melanoma_metadata.cell_line == 'UACC_62', 0],
          melanoma_pca_data[melanoma_metadata.cell_line == 'UACC_62', 1], alpha=0.4,c='tab:orange', label='UACC_62')

ax.set_xlabel('PC1 ({0:.2f}%)'.format(melanoma_pca.explained_variance_ratio_[0]*100))
ax.set_ylabel('PC2 ({0:.2f}%)'.format(melanoma_pca.explained_variance_ratio_[1]*100))

plt.legend(loc="best")
plt.show()

Note that the PC axes have been annotated with the total amount of variation in the data that has been captured by that PC. PC1 always has the most, followed by PC2, in order.

We can see that each cell line group clearly seperates from each other in the first two PCs.

We can also visualize the genes driving the PCs:


In [ ]:
#Linear transformation for PC1 and PC2, weighted by the proportion of variation explained
pc1_loadings = melanoma_pca.components_[0, :].T
pc1_weighted_loadings = pc1_loadings * melanoma_pca.explained_variance_[0]
print(pc1_weighted_loadings)

gene_names = melanoma_rnaseq.loc[:, 'A1BG':].columns.values

#Get the top genes across both PCS (sum of absolute loadings)
n_top_genes=5
top_pc1_gene_idxs = np.argsort(np.abs(pc1_weighted_loadings))[-n_top_genes:]
print(top_pc1_gene_idxs)

#Add arrows to PCA plot
for gene_idx in top_pc1_gene_idxs:
  print(gene_names[gene_idx])
  ax.annotate(text = gene_names[gene_idx],
              xy = [0, 0],
              xytext = [melanoma_pca.components_.T[gene_idx, 0] * melanoma_pca.explained_variance_[0],
                        melanoma_pca.components_.T[gene_idx, 1] * melanoma_pca.explained_variance_[1]],
              arrowprops=dict(arrowstyle='<-',linewidth=1, shrinkA=0.9))

fig

Here we can see that PC1 is associated with increased levels of HDGF, FTSJ3, SSRP1 and PFDN2, as well as lower levels of CPQ.

### <font color=brown>Hands on practice</font>
Identify and plot the top 3 genes most strongly associated with PC2

In [ ]:
gene_names[:5]

In [ ]:
#Linear transformation loadings PC2, weighted by the proportion of variation explained


#Get the top genes across both PCs (sum of absolute loadings)

#Add arrows to PCA plot


### The explained variance of each principal component

Lets look a bit more closely at the variance explained by each PC.

In [ ]:
# This shows how much of the variance is explained by each of the principal components.
print(melanoma_pca.explained_variance_ratio_)

In [ ]:
# This calculates the cumulative explained variance
cum_var_exp = np.cumsum(melanoma_pca.explained_variance_ratio_)
print(cum_var_exp)

We can make a barplot to illustrate this.


In [ ]:
# Calculate and plot the variance explained and cumulative variance explained
percent_var_expl = melanoma_pca.explained_variance_ratio_*100
cum_var_expl = np.cumsum(percent_var_expl)

# plot explained variances
fig = plt.figure()
ax = fig.add_subplot(111)
plt.bar(range(len(percent_var_expl)), percent_var_expl, alpha=0.5,
        align='center', label='% Variance captured')
plt.step(range(len(cum_var_expl)), cum_var_expl, where='mid',
         label='Cumulative % variance captured')
plt.ylabel('% Variance captured')
plt.xlabel('')
plt.legend(loc='best')
plt.xticks(range(len(percent_var_expl)),
           ["PC{0}".format(_+1) for _ in range(len(percent_var_expl))])
plt.show()




# Part 2: tSNE and UMAP for single cell gene expression

Unlike PCA, t-SNE and UMAP are non-linear probabilistic methods. Both t-SNE and UMAP calculate "distances" or "similarities" between points of the high dimensional data. Then, they attempt to force it into a set small number of dimensions (usually 2D) while retaining neighboring points in the graph close together. Both methods are very commonly used for single-cell RNAseq to visualize how cells cluster together.

#### PCA vs t-SNE/UMAP

| PCA | t-SNE / UMAP |
|-----|--------------|
| Linear transformation | Non-linear dimensionality reduction |
|Variability partitioned into <br> orthogonal PCs, the number of <br> which depends on the data | Always tries to reduce data <br> to a fixed number of dimensions|
| Deterministic <br> (the same every time) | Probabilistic (will look <br> a bit different when run again on the same data) |
| Can feed into downstream analysis | Not suitable for downstream analysis |



To illustrate t-SNE and UMAP, let's examine a sample dataset of single cell RNAseq of human lung samples originally published by Zilionas et al:

>*Zilionis R et al. (2019). Single-cell transcriptomics of human and mouse lung cancers reveals conserved myeloid populations across individuals and species. Immunity 50(5), 1317-1334.*

This dataset has been downsampled to a few hundred cells and 500 genes for ease of analysis. Cells with very low and very high (possible doublet) numbers of detected genes were removed, along with genes with high proportions of mitchondiral gene reads. Raw counts were log transformed after adding 1 to all counts.





In [ ]:
# Read the single cell Zilionis lung dataset
lung_scrnaseq = pd.read_csv('~/LECTURE_MATERIALS/DataFiles/sampled_zilionis_lung_scRNAseq_log.csv')


#### T-SNE
t-SNE calculates a similarity score using the t-distribution, where close points get high similarity scores, and far away points are in the tails of the distribution and get low similarity. Thus, far away points have very little influence over the clustering.

Points are then randomly distributed in low dimensional space, and the similarity scores recalculated for this random distribution. The t-SNE algorithm then iteratively moves points to try and make the low dimensional similarities as close as possible to the high dimensional similarities

The key parameters for t-SNE are:

 - `n_components` (default: 2): Dimension of the embedded space.
 - `perplexity` (default: 30): The perplexity is related to the expected spread of the t-distribution used to calculate the distances. **Really should be smaller than the number of points (typically between 5 and 50).**


Website resources:
 - https://distill.pub/2016/misread-tsne/
 - https://www.youtube.com/watch?v=NEaUSP4YerM
 - https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.tsne.html



In [ ]:
# Run tSNE on the Zilionis lung dataset

#run tSNE
lung_scrnaseq_tsne = TSNE(n_components=2, perplexity=50,
                                random_state=3425).fit_transform(lung_scrnaseq.loc[:,"MT-RNR2":])

#plot tSNE
celltypes = set(lung_scrnaseq.cellType)

fig = plt.figure()
ax = fig.add_subplot(111)
for celltype in celltypes:
  ax.scatter(lung_scrnaseq_tsne[lung_scrnaseq['cellType'] == celltype, 0],
             lung_scrnaseq_tsne[lung_scrnaseq['cellType'] == celltype, 1],
             alpha=0.75, marker=".", s=5, label=celltype)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(loc="best")
plt.show()


#### UMAP
UMAP is conceptually very similar to UMAP. The authors of UMAP claim that it preserves far away point distances better than t-SNE. Instead of using a t-distribution to calculate similarity, UMAP creates a fuzzy graph approach based on selecting a number of nearest neighbours.

Important parameters for UMAP include:

 - `n_components` (default: 2): Dimension of the embedded space.
 - `n_neighbours` (default: 15): This is the number of nearby points UMAP considers when creating it's fuzzy graph representation. Smaller numbers = focus on local structure, larger numbers = consider more far away points
 - `min_dist` (default 0.1): How close UMAP can pack points together in the low dimensional space


The code below calculates and plots UMAP transformations for the single cell data. We can see that both approaches separate clusters corresponding to different immune cell types well, but not perfectly.

https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.umap.html

In [ ]:
# run UMAP on the Zilionis lung dataset

# run UMAP
lung_scrnaseq_umap = UMAP().fit_transform(lung_scrnaseq.loc[:,"MT-RNR2":])

# Plot UMAP
fig = plt.figure()
ax = fig.add_subplot(111)
for celltype in celltypes:
  ax.scatter(lung_scrnaseq_umap[lung_scrnaseq['cellType'] == celltype, 0],
             lung_scrnaseq_umap[lung_scrnaseq['cellType'] == celltype, 1],
             alpha=0.75, marker=".", s=5, label=celltype)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(loc="best")
plt.show()

## <font color=blue>Bonus: Intro to `scanpy`

The `scanpy` package allows easy to use single cell data analyses in Python. It has a large array of built in functions for different analyses (data pre-processing, visualization, clustering, trajectory etc).

**Resources:**
*   https://genomebiology.biomedcentral.com/articles/10.1186/s13059-017-1382-0
*   https://scanpy.readthedocs.io/en/stable/

To demonstrate, let's examine a popular single cell RNA seq data set of human PBMC samples, used for the [SCANPy tutorial](https://scanpy-tutorials.readthedocs.io/en/latest/pbmc3k.html) and by others (10X genomics):
**pbmc_data_master.h5ad**


We will now use `scanpy` which we loaded as sc, and use tSNE and UMAP visualize the single cell PBMC data in 2D.





In [ ]:
#Read sc PBMC file to be used for scanpy

pbmc_file = "pbmc_data_master.h5ad"

pbmc_scrnaseq = sc.read_h5ad(pbmc_file)

In [ ]:
# setting up a new results file to save data into
results_file = "dat_mod.h5ad"

Note: `scanpy` functions are divided into 3 categories: preprocessing (labeled as `pp`), analysis tools (labeled as `tl`), and plotting (labeled as `pl`).

### Running PCA

In [ ]:
# Compute PCA to determine the default number of components
sc.tl.pca(pbmc_scrnaseq, svd_solver="arpack")

pbmc_scrnaseq.write(results_file)

In [ ]:
#plotting the PCA
sc.pl.pca(pbmc_scrnaseq)

Each dot represents a different cell profiled in the dataset. You can see by eye that the data appear to form separate clusters (perhaps explained by cell type similarities in expression).

Besides plotting the PC visualization, you can also plot the variance explained

In [ ]:
#plotting the PC variance explained
sc.pl.pca_variance_ratio(pbmc_scrnaseq, log=True)

### Clustering analysis
The recommended clustering method for scRNA-seq is the Leiden graph-clustering method (community detection based on optimizing modularity) [Traag et al., 2019](https://www.nature.com/articles/s41598-019-41695-z). Note that Leiden clustering directly clusters a neighborhood graph of cells.

First, we need to calculate nearest neighbors. This can be done with the `pp.neighbors()` function:

In [ ]:
#compute neighbors
#n_neighbors and n_pcs can be modified

sc.pp.neighbors(pbmc_scrnaseq, n_neighbors=10, n_pcs=40)

The Leiden clustering then uses the nearest neighbor distances calculated by `pp.neighbors()` to detect clusters.

In [ ]:
# Compute Leiden clustering

sc.tl.leiden(
    pbmc_scrnaseq,
    resolution=0.9, #impact clusters
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
)

Now, we can update the PCA visualization with the clusters computed by the Leiden method

In [ ]:
#plotting the PCA
sc.pl.pca(pbmc_scrnaseq, color = "leiden")

Note that because we performed the analysis on the `pbmc_scrnaseq` object, that object has been updated with the computed clusters.

Now, let's look at how the cells cluster by t-SNE and UMAP

In [ ]:
#tSNE

#run tSNE
sc.tl.tsne(pbmc_scrnaseq, perplexity = 40)

#plot tSNE
sc.pl.tsne(pbmc_scrnaseq, color=["leiden"])

In [ ]:
# UMAP
# Run UMAP
sc.tl.umap(pbmc_scrnaseq)

# Plot UMAP
sc.pl.umap(pbmc_scrnaseq, color=["leiden"])

We can find which genes express differently between the different Leiden clusters with `rank_genes_groups()`.

*Note:* According to the `scanpy` documentation, comparing between cells leads to highly inflated p-values, since cells are not independent observations [Squair et al., 2021]. Especially in single-cell data, consider instead to use more appropriate methods such as combining pseudobulking with `PyDESeq2` documentation.

In [ ]:
#finding marker genes

sc.tl.rank_genes_groups(pbmc_scrnaseq, "leiden", method="wilcoxon")

sc.pl.rank_genes_groups(pbmc_scrnaseq, n_genes=5, sharey=False) #graphing top 5 genes in each cluster

In [ ]:
pd.DataFrame(pbmc_scrnaseq.uns["rank_genes_groups"]["names"]).head(10)

In [ ]:
sc.pl.rank_genes_groups_heatmap(pbmc_scrnaseq, n_genes=10, groupby = "leiden", show_gene_labels=True) #graphing by heatmap top ten genes in each cluster

In [ ]:
pbmc_scrnaseq.write(results_file)

### Examining known genes of interest
You can overlay the expression levels of genes of interest onto the different clusters

In [ ]:
#Find and pull put expression of one gene

sc.pl.umap(pbmc_scrnaseq, color=["leiden", "CD19", "CD8A", "IL7R"])

In [ ]:
#define marker genes
marker_genes = [
    *["IL7R", "CD79A", "MS4A1", "CD8A", "CD8B", "LYZ", "CD14", "CD4"],
    *["LGALS3", "S100A8", "GNLY", "NKG7", "KLRB1"],
    *["FCGR3A", "MS4A7", "FCER1A", "CST3", "PPBP"],
]

In [ ]:
marker_genes

In [ ]:
sc.pl.dotplot(pbmc_scrnaseq, marker_genes, groupby="leiden");

You can redefine the cluster labels based on the expression profiles of these known genes-of-interest

In [ ]:
# Cluster by cell type

new_cluster_names = [
    "CD4 T",
    "B",
    "FCGR3A+ Monocytes",
    "NK",
    "CD8 T",
    "CD14+ Monocytes",
    "Dendritic",
    "Megakaryocytes",
]

pbmc_scrnaseq.rename_categories("leiden", new_cluster_names)


In [ ]:
sc.pl.umap(
    pbmc_scrnaseq, color="leiden", legend_loc="on data", title="", frameon=False)

## Clustering T-cells, NK cells and B-cells

If we are interested in identifying the heterogeneity within the T- and T-cell like populations, based on the assigned cluster labels from above, we can use that to subset out the cells within that cluster and repeat the clustering and UMAP on the new dataset.

In [ ]:
#subset out clusters of interest
subset_pbmc_scrnaseq = pbmc_scrnaseq[pbmc_scrnaseq.obs['leiden'].isin(['CD8 T', 'CD4 T', 'NK', 'B'])]

In [ ]:
#calculate PCA for the new subsetted data
sc.tl.pca(subset_pbmc_scrnaseq, svd_solver="arpack")

In [ ]:
#compute neighbors
sc.pp.neighbors(subset_pbmc_scrnaseq, n_neighbors=15, n_pcs=40)
subset_pbmc_scrnaseq

In [ ]:
#perform leiden clustering

sc.tl.leiden(
    subset_pbmc_scrnaseq,
    resolution=0.9,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
)

In [ ]:
# compute t-SNE
sc.tl.tsne(subset_pbmc_scrnaseq, perplexity = 30)

# Plot the clusters using leiden clustering and visualize t-SNE and key T cell marker genes
sc.pl.tsne(subset_pbmc_scrnaseq, color = ["leiden", 'CD8A', "IL7R", "NKG7", "CD19"])

In [ ]:
# Compute UMAP
sc.tl.umap(subset_pbmc_scrnaseq)

# Plot the clusters using leiden clustering and visualize UMAP and key T cell marker genes
sc.pl.umap(subset_pbmc_scrnaseq, color=["leiden", 'CD8A', "IL7R", "NKG7", "CD19"])